In [ ]:
# ============================================================
# CELL 1: CONFIG + IMPORTS
# ============================================================

import os, random, time, json, warnings
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import timm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix,
                             cohen_kappa_score, matthews_corrcoef, roc_auc_score)
warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
CFG = dict(
    # Model
    backbone="mobilenetv3_small_100",
    use_coord_attn=True,
    use_eca=True,
    use_fem=True,
    use_bifpn=True,
    
    # Training
    img_size=256,
    batch_size=48,
    epochs=40,
    lr=3e-4,
    weight_decay=1e-4,
    label_smoothing=0.1,
    dropout=0.3,
    warmup_epochs=3,
    patience=8,
    
    # Cross-validation
    n_folds=5,
    folds_to_run=[0, 1, 2, 3, 4],
    sim_threshold=0.05,
    
    # Others
    seed=42,
)

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(CFG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

OUT_DIR = "/kaggle/working/output" if os.path.exists("/kaggle/working") else "./output"
os.makedirs(OUT_DIR, exist_ok=True)

print("Device :", device)
print("Output :", OUT_DIR)
print("Config :", CFG)

In [ ]:
# ============================================================
# CELL 2: LOAD DATA (Auto-detect classes from folders)
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# -------------------- Config --------------------
DATA_ROOT = "/kaggle/input/datasets/sabbir4724/dentiro1"   # ← Dataset path
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")
S = 256

# Auto-detect root if needed
def find_root(base="/kaggle/input"):
    for d, subs, _ in os.walk(base):
        
        if len([s for s in subs if not s.startswith('.')]) >= 2:
            return d
    raise FileNotFoundError("Class folders not found. Set DATA_ROOT manually.")

ROOT = DATA_ROOT or find_root()
print("Data root:", ROOT)

# -------------------- Auto-detect CLASSES from folders --------------------
CLASSES = sorted([
    d for d in os.listdir(ROOT)
    if os.path.isdir(os.path.join(ROOT, d)) and not d.startswith('.')
])
print("Detected Classes:", CLASSES)

# CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"Cannot read {path}")
    img = cv2.resize(img, (S, S), interpolation=cv2.INTER_AREA)
    img = clahe.apply(img)
    return img

# -------------------- Load all images --------------------
paths, labels = [], []

for ci, c in enumerate(CLASSES):
    folder = os.path.join(ROOT, c)
    files = [f for f in sorted(os.listdir(folder)) if f.lower().endswith(IMG_EXT)]
    
    for f in files:
        paths.append(os.path.join(folder, f))
        labels.append(ci)

y = np.array(labels)
X = np.stack([load_gray(p) for p in tqdm(paths, desc="Loading Images")])

print("\nX shape:", X.shape, "| dtype:", X.dtype)
print("Memory  : %.1f MB" % (X.nbytes / 1e6))
print("\nClass distribution:")
print(pd.Series(y).map(dict(enumerate(CLASSES))).value_counts().sort_index())

In [ ]:
# CELL 3: SIMILARITY GROUPING + STRATIFIED GROUP K-FOLD
# Patient ID is unavailable, so near-duplicate / same-patient images are grouped by
# embedding similarity and kept together in the same fold (reduces leakage).
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def extract_embeddings(X, batch=128):
    m = timm.create_model("mobilenetv3_large_100", pretrained=True, num_classes=0).to(device).eval()
    out = []
    for i in tqdm(range(0, len(X), batch), desc="Embeddings"):
        xb = torch.from_numpy(X[i:i + batch]).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
        xb = ((xb - MEAN) / STD).to(device)
        out.append(F.normalize(m(xb), dim=1).cpu())
    del m
    return torch.cat(out).numpy()

E = extract_embeddings(X)
D = np.clip(1.0 - E @ E.T, 0, None).astype(np.float32)
np.fill_diagonal(D, 0); D = (D + D.T) / 2
Z = linkage(squareform(D, checks=False), method="average")

print("Threshold sweep (pick one where the largest group stays small, e.g. < 2% of data):")
for t in [0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.2]:
    g = fcluster(Z, t, criterion="distance"); sz = np.bincount(g)[1:]
    print(f"  t={t:<5} groups={len(sz):>5}  largest={sz.max():>5}  imgs_in_multi_groups={(sz[sz>1]).sum():>5}")

groups = fcluster(Z, CFG["sim_threshold"], criterion="distance")
print(f"\nUsing t={CFG['sim_threshold']}: {groups.max()} groups, largest={np.bincount(groups).max()}")

sgkf = StratifiedGroupKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
folds = []
for trval_idx, test_idx in sgkf.split(X, y, groups):
    inner = StratifiedGroupKFold(n_splits=8, shuffle=True, random_state=CFG["seed"])
    tr_i, va_i = next(inner.split(trval_idx, y[trval_idx], groups[trval_idx]))
    folds.append(dict(train=trval_idx[tr_i], val=trval_idx[va_i], test=test_idx))

for i, f in enumerate(folds):
    cnt = lambda idx: np.bincount(y[idx], minlength=len(CLASSES)).tolist()
    print(f"Fold {i}: train={len(f['train'])} {cnt(f['train'])} | val={len(f['val'])} {cnt(f['val'])} | test={len(f['test'])} {cnt(f['test'])}")

pd.DataFrame(dict(path=paths, label=y, group=groups)).to_csv(os.path.join(OUT_DIR, "groups.csv"), index=False)

In [ ]:
# ============================================================
# CELL 4: DATASET + TRANSFORMS
# ============================================================
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset

MEAN_ = [0.485, 0.456, 0.406]
STD_  = [0.229, 0.224, 0.225]

# Training: mild geometric + intensity augmentation
train_tf = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN_, STD_),
])


eval_tf = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(MEAN_, STD_),
])

class DentalDS(Dataset):
    def __init__(self, X, y, idx, tf):
        self.X, self.y, self.idx, self.tf = X, y, np.asarray(idx), tf
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        img = Image.fromarray(self.X[j])
        return self.tf(img), int(self.y[j])

# Quick sanity check
_ds = DentalDS(X, y, folds[0]["train"], train_tf)
_x, _y = _ds[0]
print("Sample:", _x.shape, _x.dtype, "| label:", _y, "| train size:", len(_ds))

In [ ]:
# ============================================================
# FULL CELL
# 5-Fold CV + Per-fold results + All curves

# ============================================================
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc,
                             accuracy_score, f1_score, precision_score, recall_score)
from sklearn.preprocessing import label_binarize
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import timm
warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
CFG.update({
    "backbone": "mobilenetv3_small_100",
    "use_coord_attn": True,
    "use_eca": True,
    "use_fem": True,
    "use_bifpn": True,
    "batch_size": 48,
    "epochs": 40,
    "lr": 3e-4,
    "warmup_epochs": 3,
    "dropout": 0.3,
    "folds_to_run": [0, 1, 2, 3, 4],
})

# -------------------- FIXED DATALOADER (warning fix) --------------------
def make_loader(idx, tf, shuffle):
    """num_workers=0 → completely removes the MultiProcessingDataLoaderIter warning"""
    return DataLoader(
        DentalDS(X, y, idx, tf),
        batch_size=CFG["batch_size"],
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        drop_last=shuffle
    )

# -------------------- MODEL --------------------
class ECA(nn.Module):
    def __init__(self, c, k=3):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
    def forward(self, x):
        y = self.avg(x).squeeze(-1).transpose(1, 2)
        y = self.conv(y).transpose(1, 2).unsqueeze(-1)
        return x * torch.sigmoid(y)

class CoordAttn(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 8)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1 = nn.Conv2d(c, h, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(h)
        self.act = nn.ReLU(inplace=True)
        self.conv_h = nn.Conv2d(h, c, 1, bias=False)
        self.conv_w = nn.Conv2d(h, c, 1, bias=False)
    def forward(self, x):
        n, c, h, w = x.shape
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)
        return x * torch.sigmoid(self.conv_h(y_h)) * torch.sigmoid(self.conv_w(y_w))

class FrequencyEnhancement(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.high = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, groups=c, bias=False),
            nn.BatchNorm2d(c), nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 1, bias=False)
        )
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c, 1), nn.Sigmoid())
    def forward(self, x):
        return x + self.high(x) * self.gate(x)

class BiFPNFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.w = nn.Parameter(torch.ones(len(channels)))
        self.eps = 1e-4
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, channels[0], 1, bias=False),
                nn.BatchNorm2d(channels[0]),
                nn.ReLU(inplace=True)
            ) for c in channels
        ])
    def forward(self, feats):
        target = feats[0].shape[-2:]
        resized = []
        for i, f in enumerate(feats):
            f = self.convs[i](f)
            if f.shape[-2:] != target:
                f = F.interpolate(f, size=target, mode="bilinear", align_corners=False)
            resized.append(f)
        w = F.relu(self.w)
        w = w / (w.sum() + self.eps)
        return sum(wi * fi for wi, fi in zip(w, resized))

class DentalNet(nn.Module):
    def __init__(self, backbone="mobilenetv3_small_100", num_classes=4,
                 use_coord=True, use_eca=True, use_fem=True, use_bifpn=True,
                 dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, features_only=True)
        chs = self.backbone.feature_info.channels()[-3:]

        self.fems = nn.ModuleList([FrequencyEnhancement(c) if use_fem else nn.Identity() for c in chs])
        self.attns = nn.ModuleList()
        for c in chs:
            layers = []
            if use_coord: layers.append(CoordAttn(c))
            if use_eca:   layers.append(ECA(c))
            self.attns.append(nn.Sequential(*layers) if layers else nn.Identity())

        self.use_bifpn = use_bifpn
        if use_bifpn:
            self.fusion = BiFPNFusion(chs)
            fuse_ch = chs[0]
        else:
            fuse_ch = sum(chs)

        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(fuse_ch * 2, num_classes)

    def forward_feats(self, x):
        feats = self.backbone(x)[-3:]
        feats = [fem(f) for fem, f in zip(self.fems, feats)]
        feats = [attn(f) for attn, f in zip(self.attns, feats)]
        return feats

    def classify(self, feats):
        if self.use_bifpn:
            fused = self.fusion(feats)
            pooled = torch.cat([
                F.adaptive_avg_pool2d(fused, 1).flatten(1),
                F.adaptive_max_pool2d(fused, 1).flatten(1)
            ], dim=1)
        else:
            pooled = torch.cat([
                torch.cat([
                    F.adaptive_avg_pool2d(f, 1).flatten(1),
                    F.adaptive_max_pool2d(f, 1).flatten(1)
                ], dim=1) for f in feats
            ], dim=1)
        return self.fc(self.drop(pooled))

    def forward(self, x):
        return self.classify(self.forward_feats(x))

def build_model(pretrained=True):
    return DentalNet(
        backbone=CFG["backbone"],
        num_classes=len(CLASSES),
        use_coord=CFG["use_coord_attn"],
        use_eca=CFG["use_eca"],
        use_fem=CFG["use_fem"],
        use_bifpn=CFG["use_bifpn"],
        dropout=CFG["dropout"],
        pretrained=pretrained
    ).to(device)

# -------------------- TRAIN / EVAL --------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(xb)
        P.append(out.float().softmax(1).cpu())
        Y.append(yb)
    P = torch.cat(P).numpy()
    Y = torch.cat(Y).numpy()
    return P, Y, P.argmax(1)

def train_one_fold(fold_id):
    fd = folds[fold_id]
    tr_loader = make_loader(fd["train"], train_tf, True)
    va_loader = make_loader(fd["val"],   eval_tf, False)

    cnt = np.bincount(y[fd["train"]], minlength=len(CLASSES))
    w = torch.tensor(len(fd["train"]) / (len(CLASSES) * cnt + 1e-6), dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["label_smoothing"])

    model = build_model()
    head_params = (list(model.fems.parameters()) + list(model.attns.parameters()) +
                   list(model.fc.parameters()))
    if CFG["use_bifpn"]:
        head_params += list(model.fusion.parameters())

    opt = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": CFG["lr"]},
        {"params": head_params, "lr": CFG["lr"] * 3},
    ], weight_decay=CFG["weight_decay"])

    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=CFG["warmup_epochs"]),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(CFG["epochs"] - CFG["warmup_epochs"], 1))
    ], milestones=[CFG["warmup_epochs"]])

    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best_f1, hist = -1.0, []
    ckpt = os.path.join(OUT_DIR, f"best_proposed_fold{fold_id}.pth")

    for ep in range(CFG["epochs"]):
        model.train()
        run_loss, n = 0.0, 0
        for xb, yb in tqdm(tr_loader, desc=f"Fold {fold_id} | Ep {ep+1}/{CFG['epochs']}", leave=False):
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(xb).float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item() * len(yb)
            n += len(yb)
        sched.step()

        # validation
        probs, labels, preds = evaluate(model, va_loader)
        val_acc = accuracy_score(labels, preds)
        val_f1  = f1_score(labels, preds, average="macro")
        hist.append({
            "epoch": ep+1,
            "train_loss": run_loss / n,
            "val_loss": None,
            "val_acc": val_acc,
            "val_f1": val_f1
        })

        flag = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), ckpt)
            flag = "  ✅"
        print(f"Fold {fold_id} | Ep {ep+1:02d} | loss {run_loss/n:.4f} | val_acc {val_acc:.4f} | macroF1 {val_f1:.4f}{flag}")

    model.load_state_dict(torch.load(ckpt, map_location=device))
    return model, pd.DataFrame(hist)

# -------------------- PER-CLASS METRICS FUNCTION --------------------
def per_class_metrics(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    rows = []
    total = cm.sum()
    for i, name in enumerate(class_names):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = total - (TP + FN + FP)
        acc  = (TP + TN) / total * 100
        tpr  = TP / (TP + FN + 1e-8) * 100
        fnr  = FN / (TP + FN + 1e-8) * 100
        fpr  = FP / (FP + TN + 1e-8) * 100
        tnr  = TN / (TN + FP + 1e-8) * 100
        prec = TP / (TP + FP + 1e-8) * 100
        f1   = 2 * prec * tpr / (prec + tpr + 1e-8)
        err  = 100 - acc
        rows.append([name, acc, tpr, fnr, fpr, tnr, prec, f1, err])
    df = pd.DataFrame(rows, columns=[
        "Class", "Accuracy (%)", "TPR (%)", "FNR (%)", "FPR (%)",
        "TNR (%)", "Precision (%)", "F1 Score (%)", "Error Rate (%)"
    ])
    return df.round(2)

# ============================================================
# RUN ALL 5 FOLDS
# ============================================================
all_results = []
all_hists = {}
all_cms = {}
all_probs = {}
all_labels = {}
all_preds = {}

for fid in CFG["folds_to_run"]:
    print(f"\n{'='*70}")
    print(f"🚀 Starting Fold {fid}")
    print(f"{'='*70}")
    
    model, hist = train_one_fold(fid)
    all_hists[fid] = hist
    
    # Test set
    te_loader = make_loader(folds[fid]["test"], eval_tf, False)
    probs, labels, preds = evaluate(model, te_loader)
    
    all_probs[fid] = probs
    all_labels[fid] = labels
    all_preds[fid] = preds
    all_cms[fid] = confusion_matrix(labels, preds)
    
    # Per-class metrics
    metrics_df = per_class_metrics(labels, preds, CLASSES)
    print(f"\n----- Fold {fid} Per-Class Metrics -----")
    print(metrics_df.to_string(index=False))
    metrics_df.to_csv(os.path.join(OUT_DIR, f"per_class_metrics_fold{fid}.csv"), index=False)
    hist.to_csv(os.path.join(OUT_DIR, f"history_fold{fid}.csv"), index=False)
    
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    
    all_results.append({
        "fold": fid,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    })
    print(f"\nFold {fid} Summary → Acc: {acc:.4f} | Macro-F1: {macro_f1:.4f} | Weighted-F1: {weighted_f1:.4f}")

# -------------------- CV SUMMARY --------------------
res_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print("5-FOLD CROSS-VALIDATION SUMMARY")
print("="*70)
print(res_df.to_string(index=False))
print("-"*70)
print(f"Mean Accuracy    : {res_df['accuracy'].mean():.4f} ± {res_df['accuracy'].std():.4f}")
print(f"Mean Macro-F1    : {res_df['macro_f1'].mean():.4f} ± {res_df['macro_f1'].std():.4f}")
print(f"Mean Weighted-F1 : {res_df['weighted_f1'].mean():.4f} ± {res_df['weighted_f1'].std():.4f}")
res_df.to_csv(os.path.join(OUT_DIR, "cv_summary.csv"), index=False)

# ============================================================
# PLOTS: 5-Fold Training / Validation Curves + Overall
# ============================================================
fig = plt.figure(figsize=(20, 16))

# ----- 1. All Folds Training Loss -----
ax1 = fig.add_subplot(2, 3, 1)
for fid in CFG["folds_to_run"]:
    h = all_hists[fid]
    ax1.plot(h["epoch"], h["train_loss"], lw=2, label=f"Fold {fid}")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training Loss (All Folds)"); ax1.legend(); ax1.grid(True, alpha=0.3)

# ----- 2. All Folds Validation Accuracy -----
ax2 = fig.add_subplot(2, 3, 2)
for fid in CFG["folds_to_run"]:
    h = all_hists[fid]
    ax2.plot(h["epoch"], h["val_acc"], lw=2, label=f"Fold {fid}")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy (All Folds)"); ax2.legend(); ax2.grid(True, alpha=0.3)

# ----- 3. All Folds Validation Macro-F1 -----
ax3 = fig.add_subplot(2, 3, 3)
for fid in CFG["folds_to_run"]:
    h = all_hists[fid]
    ax3.plot(h["epoch"], h["val_f1"], lw=2, label=f"Fold {fid}")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Macro-F1")
ax3.set_title("Validation Macro-F1 (All Folds)"); ax3.legend(); ax3.grid(True, alpha=0.3)

# ----- 4. Mean ± Std Curves -----
ax4 = fig.add_subplot(2, 3, 4)
max_ep = max(len(all_hists[f]) for f in CFG["folds_to_run"])
epochs = np.arange(1, max_ep + 1)

def mean_std(key):
    vals = []
    for f in CFG["folds_to_run"]:
        h = all_hists[f]
        v = h[key].values
        if len(v) < max_ep:
            v = np.pad(v, (0, max_ep - len(v)), constant_values=np.nan)
        vals.append(v)
    vals = np.array(vals)
    return np.nanmean(vals, axis=0), np.nanstd(vals, axis=0)

mean_loss, std_loss = mean_std("train_loss")
mean_acc, std_acc   = mean_std("val_acc")
mean_f1, std_f1     = mean_std("val_f1")

ax4.plot(epochs, mean_loss, "b-", lw=2, label="Train Loss (mean)")
ax4.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, alpha=0.2, color="b")
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Loss")
ax4.set_title("Mean Training Loss ± Std"); ax4.legend(); ax4.grid(True, alpha=0.3)

# ----- 5. Mean Val Acc & F1 -----
ax5 = fig.add_subplot(2, 3, 5)
ax5.plot(epochs, mean_acc, "g-", lw=2, label="Val Acc (mean)")
ax5.fill_between(epochs, mean_acc - std_acc, mean_acc + std_acc, alpha=0.2, color="g")
ax5.plot(epochs, mean_f1, color="orange", lw=2, label="Val Macro-F1 (mean)")
ax5.fill_between(epochs, mean_f1 - std_f1, mean_f1 + std_f1, alpha=0.2, color="orange")
ax5.set_xlabel("Epoch"); ax5.set_ylabel("Score")
ax5.set_title("Mean Validation Acc & F1 ± Std"); ax5.legend(); ax5.grid(True, alpha=0.3)

# ----- 6. 5-Fold Accuracy Bar -----
ax6 = fig.add_subplot(2, 3, 6)
colors = plt.cm.Set2(np.linspace(0, 1, 5))
bars = ax6.bar([f"Fold {f}" for f in res_df["fold"]], res_df["accuracy"] * 100,
               color=colors, edgecolor="black")
ax6.axhline(res_df["accuracy"].mean() * 100, color="red", linestyle="--", lw=2, label=f"Mean = {res_df['accuracy'].mean()*100:.2f}%")
ax6.set_ylabel("Accuracy (%)")
ax6.set_title("Test Accuracy per Fold")
ax6.set_ylim(0, 105)
for bar, val in zip(bars, res_df["accuracy"] * 100):
    ax6.text(bar.get_x() + bar.get_width()/2, val + 1, f"{val:.1f}", ha="center", fontsize=10)
ax6.legend(); ax6.grid(True, axis="y", alpha=0.3)

plt.suptitle("DentAttn-MSFNet (MobileNetV3-Small) | 5-Fold Cross-Validation",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "5fold_curves.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ All 5 folds completed!")
print(f"✅ Results saved to: {OUT_DIR}")
print(f"   - cv_summary.csv")
print(f"   - per_class_metrics_fold0~4.csv")
print(f"   - history_fold0~4.csv")
print(f"   - 5fold_curves.png")
print(f"   - best_proposed_fold0~4.pth")

In [ ]:
import os
import torch

MODEL_CFG = dict(
    backbone="mobilenetv3_small_100",
    use_coord=True, use_eca=True, use_fem=True, use_bifpn=True,
    dropout=0.3,
)

def export_model(state_dict_path, out_prefix, class_names, img_size=224,
                 model_cfg=MODEL_CFG, device="cpu"):
    model = DentalNet(num_classes=len(class_names), pretrained=False, **model_cfg)
    model.load_state_dict(torch.load(state_dict_path, map_location="cpu"))
    model.eval().to(device)

    ckpt = {
        "state_dict": model.state_dict(),
        "model_cfg": model_cfg,
        "class_names": list(class_names),
        "num_classes": len(class_names),
        "img_size": img_size,
    }
    p1 = f"{out_prefix}.pth"
    torch.save(ckpt, p1)

    p2 = f"{out_prefix}_full.pth"
    torch.save(model, p2)

    p3 = f"{out_prefix}.pt"
    example = torch.randn(1, 3, img_size, img_size, device=device)
    with torch.no_grad():
        traced = torch.jit.trace(model, example, check_trace=False)
    traced.save(p3)

    print("Saved:\n ", p1, "\n ", p2, "\n ", p3)
    return p1, p2, p3

def load_proposed(ckpt_path, device="cpu"):
    ck = torch.load(ckpt_path, map_location=device)
    model = DentalNet(num_classes=ck["num_classes"], pretrained=False, **ck["model_cfg"])
    model.load_state_dict(ck["state_dict"])
    return model.to(device).eval(), ck["class_names"]

def load_full(full_path, device="cpu"):
    return torch.load(full_path, map_location=device, weights_only=False).eval()

def load_torchscript(pt_path, device="cpu"):
    return torch.jit.load(pt_path, map_location=device).eval()

print("OK:", export_model, load_proposed, load_full, load_torchscript)

In [ ]:
best_fold = int(res_df.loc[res_df["macro_f1"].idxmax(), "fold"])
sd_path   = os.path.join(OUT_DIR, f"best_proposed_fold{best_fold}.pth")
prefix    = os.path.join(OUT_DIR, "DentAttn_MSFNet_final")

export_model(sd_path, prefix, CLASSES, img_size=224, device=device)

# ---- verify ----
x = torch.randn(1, 3, 224, 224).to(device)
m1, classes = load_proposed(prefix + ".pth", device)
m2 = load_full(prefix + "_full.pth", device)
m3 = load_torchscript(prefix + ".pt", device)
with torch.no_grad():
    print(torch.allclose(m1(x), m2(x), atol=1e-5),
          torch.allclose(m1(x), m3(x), atol=1e-4))
print("Classes:", classes)

In [ ]:
# ============================================================
# POST-TRAINING PLOTS: Confusion Matrix + ROC-AUC + t-SNE
# Run this in a NEW cell AFTER the 5-fold training cell finishes.

# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

FOLDS = CFG["folds_to_run"]
n_cls = len(CLASSES)

# ------------------------------------------------------------
# 1. CONFUSION MATRICES
# ------------------------------------------------------------
# 1a. Per-fold (counts)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.ravel()
for i, fid in enumerate(FOLDS):
    sns.heatmap(all_cms[fid], annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[i])
    axes[i].set_title(f"Fold {fid}")
    axes[i].set_xlabel("Predicted"); axes[i].set_ylabel("True")

# 6th panel: pooled across all folds
cm_sum = sum(all_cms[f] for f in FOLDS)
sns.heatmap(cm_sum, annot=True, fmt="d", cmap="Greens", cbar=False,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[5])
axes[5].set_title("All Folds (Summed)")
axes[5].set_xlabel("Predicted"); axes[5].set_ylabel("True")
for j in range(len(FOLDS), 5):
    axes[j].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrices_per_fold.png"), dpi=300, bbox_inches="tight")
plt.show()

# 1b. Pooled, counts + row-normalized (paper figure)
cm_norm = cm_sum / cm_sum.sum(axis=1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm_sum, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Confusion Matrix (Counts, 5-Fold Pooled)")
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", vmin=0, vmax=1,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title("Confusion Matrix (Normalized by True Class)")
for ax in axes:
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix_pooled.png"), dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 2. ROC-AUC (One-vs-Rest)
# ------------------------------------------------------------
# 2a. Pooled across all folds, per-class + macro
probs_all = np.concatenate([all_probs[f] for f in FOLDS])
labels_all = np.concatenate([all_labels[f] for f in FOLDS])
Y_bin = label_binarize(labels_all, classes=list(range(n_cls)))

fpr, tpr, roc_auc = {}, {}, {}
for c in range(n_cls):
    fpr[c], tpr[c], _ = roc_curve(Y_bin[:, c], probs_all[:, c])
    roc_auc[c] = auc(fpr[c], tpr[c])

# macro-average
grid = np.linspace(0, 1, 500)
mean_tpr = np.mean([np.interp(grid, fpr[c], tpr[c]) for c in range(n_cls)], axis=0)
macro_auc = auc(grid, mean_tpr)

plt.figure(figsize=(8, 7))
colors = plt.cm.tab10(np.arange(n_cls))
for c in range(n_cls):
    plt.plot(fpr[c], tpr[c], color=colors[c], lw=2,
             label=f"{CLASSES[c]} (AUC = {roc_auc[c]:.4f})")
plt.plot(grid, mean_tpr, "k--", lw=2.5, label=f"Macro-average (AUC = {macro_auc:.4f})")
plt.plot([0, 1], [0, 1], "grey", lw=1, linestyle=":")
plt.xlim(0, 1); plt.ylim(0, 1.02)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curves (One-vs-Rest, 5-Fold Pooled)")
plt.legend(loc="lower right"); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_pooled.png"), dpi=300, bbox_inches="tight")
plt.show()

# 2b. Per-fold macro AUC table (mean ± std for the paper)
rows = []
for f in FOLDS:
    yb = label_binarize(all_labels[f], classes=list(range(n_cls)))
    row = {"fold": f}
    aucs = []
    for c in range(n_cls):
        fp, tp, _ = roc_curve(yb[:, c], all_probs[f][:, c])
        row[CLASSES[c]] = auc(fp, tp)
        aucs.append(row[CLASSES[c]])
    row["macro_auc"] = np.mean(aucs)
    rows.append(row)
auc_df = pd.DataFrame(rows)
print("\nPer-fold AUC:")
print(auc_df.round(4).to_string(index=False))
print(f"\nMacro AUC (mean ± std over folds): "
      f"{auc_df['macro_auc'].mean():.4f} ± {auc_df['macro_auc'].std():.4f}")
auc_df.to_csv(os.path.join(OUT_DIR, "auc_per_fold.csv"), index=False)

# ------------------------------------------------------------
# 3. t-SNE (features from the LAST fold's best model, on its test set)
#    `model` still holds the last fold's model after the training loop.
# ------------------------------------------------------------
last_fold = FOLDS[-1]
te_loader = make_loader(folds[last_fold]["test"], eval_tf, False)

# capture the pooled feature vector that goes into the final classifier
captured = []
hook = model.fc.register_forward_hook(lambda m, inp, out: captured.append(inp[0].detach().cpu()))

model.eval()
Ys = []
with torch.no_grad():
    for xb, yb in te_loader:
        model(xb.to(device, non_blocking=True))
        Ys.append(yb)
hook.remove()

feats = torch.cat(captured).numpy()
Ys = torch.cat(Ys).numpy()

perp = min(30, max(5, (len(feats) - 1) // 3))
emb = TSNE(n_components=2, perplexity=perp, init="pca",
           learning_rate="auto", random_state=42).fit_transform(feats)

plt.figure(figsize=(8, 7))
for c in range(n_cls):
    m = Ys == c
    plt.scatter(emb[m, 0], emb[m, 1], s=25, alpha=0.8, color=colors[c], label=CLASSES[c])
plt.title(f"t-SNE of Learned Features (Fold {last_fold} Test Set)")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tsne_features.png"), dpi=300, bbox_inches="tight")
plt.show()

print("\n✅ Saved: confusion_matrices_per_fold.png, confusion_matrix_pooled.png,")
print("          roc_pooled.png, auc_per_fold.csv, tsne_features.png")